# Uncertainty-Based Selective Prediction for MedCLIPSeg & Modified Models

**ELEC 6661 — Medical Image Processing**, Concordia University (IMPACT Lab)  
**Primary Reference Baseline:** MedCLIPSeg (*Koleilat et al., CVPR 2026*)  
**Primary Uncertainty Metric:** Foreground Mean Predictive Entropy ($U_{\mathrm{fg}}$)  
**Frozen Operating Threshold:** $\tau = 0.064058324$ (Calibrated on BUSI Validation)  

---

## Purpose & Capabilities
This notebook provides a complete, model-independent evaluation pipeline to quantify the performance of **selective prediction** (automated quality triage with an abstention option) for medical image segmentation.

Under clinical domain shift, deep segmentation models can produce confident, silent errors. By aggregating dense predictive entropy strictly over the predicted lesion ($U_{\mathrm{fg}}$), this framework:
1. **Eliminates Background Dilution:** Decouples uncertainty estimation from relative tumor area.
2. **Triages Predictions:** Retains reliable outputs for autonomous diagnostic review while routing ambiguous cases to expert sonographers.
3. **Evaluates Risk-Coverage Efficiency:** Quantifies Area Under the Risk-Coverage Curve (AURC) alongside **Random Baseline** and **Empirical Oracle** bounds.

### Supported Evaluation Modes:
- **Mode 1 (Precomputed Predictions / CSVs):** Load existing segmentation CSV tables containing image IDs, Dice coefficients, and uncertainty scores.
- **Mode 2 (Raw Mask & Entropy Arrays):** Load binary predicted masks, ground truth masks, and raw `.npy` predictive entropy maps.
- **Mode 3 (Synthetic Demonstration Fallback):** Automatically runs a demonstration cohort if no external files are provided, allowing instant 1-click execution.

In [ ]:
# Cell 2: Environment Setup & Tool Dependency Loading
import sys
import os
from pathlib import Path

# Install minimal dependencies if running in hosted Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("[INFO] Running inside Google Colab environment.")
    !pip install -q numpy scipy pandas matplotlib pillow
except ImportError:
    IN_COLAB = False
    print("[INFO] Running in local Jupyter / Python environment.")

import numpy as np
import pandas as pd
import scipy.ndimage as ndi
from scipy.stats import spearmanr
import matplotlib
# Use non-interactive backend when running headless
if not IN_COLAB and 'IPython' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import json
from typing import Dict, List, Optional, Tuple, Union

# Configure matplotlib styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120

def resolve_existing_path(path_str: str) -> str:
    """Search common workspace relative prefixes so paths work across local and Colab environments."""
    if not path_str:
        return ""
    if os.path.exists(path_str):
        return path_str
    candidates = [
        os.path.join("..", path_str),
        os.path.join("..", "..", path_str),
        path_str.replace("project/data/outputs/", "../data/outputs/"),
        path_str.replace("project/data/outputs/", "../../data/outputs/"),
        path_str.replace("project/data/outputs/", "data/outputs/"),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return path_str

print(f"Setup complete. Python {sys.version.split()[0]} ready.")

In [ ]:
# =============================================================================
# ⚙️ CELL 3: USER CONFIGURATION — EDIT THIS CELL ONLY
# =============================================================================
# All remaining cells execute autonomously without requiring source-code edits.

# --- 1. Dataset Evaluation Settings ---
# Specify either 'csv_path' (precomputed predictions) OR mask/map directories.
# If files do not exist at the specified path, synthetic demonstration data is used.
DATASETS = {
    "BUSI_VAL": {
        "enabled": True,
        "name": "BUSI Validation (N=65 Lesion)",
        "csv_path": "project/data/outputs/BUSI/validation/seed666/aggregation_results/uncertainty_scores_BUSI_val_seed666.csv",
        "is_busi": True,
        "split": "val",
    },
    "BUSI_TEST": {
        "enabled": True,
        "name": "BUSI Test (N=65 Lesion / N=78 Full)",
        "csv_path": "project/data/outputs/BUSI/aggregation_results/seed666/uncertainty_scores_BUSI_test_seed666.csv",
        "is_busi": True,
        "split": "test",
    },
    "UDIAT": {
        "enabled": True,
        "name": "UDIAT (Spain, N=25)",
        "csv_path": "project/data/outputs/UDIAT/aggregation_results/seed666/uncertainty_scores_UDIAT_test_seed666.csv",
        "is_busi": False,
    },
    "BUID": {
        "enabled": True,
        "name": "BUID (Academic, N=35)",
        "csv_path": "project/data/outputs/BUID/aggregation_results/seed666/uncertainty_scores_BUID_test_seed666.csv",
        "is_busi": False,
    },
    "BUSUC": {
        "enabled": True,
        "name": "BUSUC (Multi-center, N=122)",
        "csv_path": "project/data/outputs/BUSUC/aggregation_results/seed666/uncertainty_scores_BUSUC_test_seed666.csv",
        "is_busi": False,
    },
    "BUSBRA": {
        "enabled": True,
        "name": "BUSBRA (Brazil, N=282)",
        "csv_path": "project/data/outputs/BUSBRA/aggregation_results/seed666/uncertainty_scores_BUSBRA_test_seed666.csv",
        "is_busi": False,
    },
}

# --- 2. Selective Prediction Operating Parameters ---
FROZEN_THRESHOLD = 0.064058324  # Authoritative frozen decision threshold tau
PRIMARY_AGGREGATOR = "u_fg"     # Primary metric (options: u_fg, u_std_fg, u_boundary, u_mean, u_ratio)
ENABLE_TWO_STAGE_SCREENING = True  # Routes empty predictions via U_mean <= 1e-4
EMPTY_FALLBACK_THRESHOLD = 1e-4
RECALIBRATE_ON_VAL = False      # If True, computes empirical tau on BUSI_VAL for TARGET_COVERAGE
TARGET_VAL_COVERAGE = 0.80      # Target nominal coverage if recalibrating

# --- 3. Optional Model Comparison (Original vs. Modified Model) ---
REFERENCE_BASELINE_CSV = ""      # Optional path to official baseline CSV for paired comparison
MODIFIED_MODEL_LABEL = "Modified Model"
BASELINE_MODEL_LABEL = "Original MedCLIPSeg"

# --- 4. Export Configuration ---
OUTPUT_DIR = "./selective_evaluation_results"
EXPORT_FIGURES_PDF = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Configuration locked. Outputs will be saved to: {OUTPUT_DIR}")

In [ ]:
# Cell 4: Input Loading, File Integrity & Validation

def validate_and_load_dataset(dset_key: str, cfg: Dict) -> pd.DataFrame:
    """Load dataset from CSV or raw arrays with strict validation checks."""
    raw_csv = cfg.get("csv_path", "")
    resolved_csv = resolve_existing_path(raw_csv)
    
    # Check if precomputed CSV exists
    if resolved_csv and os.path.exists(resolved_csv):
        df = pd.read_csv(resolved_csv)
        print(f"[LOADED] {cfg['name']} from {resolved_csv} ({len(df)} cases).")
        
        # Standardize column naming case-insensitively
        col_map = {c.lower(): c for c in df.columns}
        if "dsc" in col_map:
            df["DSC"] = df[col_map["dsc"]].astype(float)
        if "nsd" in col_map:
            df["NSD"] = df[col_map["nsd"]].astype(float)
            
        # Standardize uncertainty columns to both lowercase and uppercase aliases
        unc_cols = ["u_fg", "u_std_fg", "u_boundary", "u_mean", "u_positive", "u_threshold", "u_ratio", "pred_area"]
        for target in unc_cols:
            if target in col_map:
                df[target] = df[col_map[target]]
                df[target.upper()] = df[col_map[target]]
            
        # Validate range
        if "DSC" in df.columns:
            assert df["DSC"].between(0.0, 1.0).all(), f"DSC values outside [0, 1] in {dset_key}!"
        return df

    # Fallback: Check if directories were provided
    pred_dir = resolve_existing_path(cfg.get("pred_mask_dir", ""))
    gt_dir = resolve_existing_path(cfg.get("gt_mask_dir", ""))
    unc_dir = resolve_existing_path(cfg.get("unc_map_dir", ""))
    
    if pred_dir and os.path.exists(pred_dir) and gt_dir and os.path.exists(gt_dir) and unc_dir and os.path.exists(unc_dir):
        print(f"[INFO] Loading raw masks and uncertainty maps from {pred_dir}...")
        records = []
        for fname in sorted(os.listdir(pred_dir)):
            if not fname.endswith(('.png', '.tif', '.jpg')):
                continue
            case_id = os.path.splitext(fname)[0]
            p_path = os.path.join(pred_dir, fname)
            g_path = os.path.join(gt_dir, fname)
            u_path = os.path.join(unc_dir, f"{case_id}.npy")
            
            if not (os.path.exists(g_path) and os.path.exists(u_path)):
                continue
                
            pred_mask = np.array(Image.open(p_path).convert('L')) > 127
            gt_mask = np.array(Image.open(g_path).convert('L')) > 127
            unc_map = np.load(u_path).astype(np.float32)
            
            # Strict dimensionality and range validation
            if pred_mask.shape != gt_mask.shape or pred_mask.shape != unc_map.shape:
                raise ValueError(f"Dimension mismatch for {case_id}: Pred {pred_mask.shape}, GT {gt_mask.shape}, Map {unc_map.shape}!")
            if np.max(unc_map) > 1.0:
                print(f"[WARNING] High entropy detected ({np.max(unc_map):.3f}) for {case_id}. Check if normalized.")
                
            dsc_val = compute_dice_local(pred_mask, gt_mask)
            u_fg_val = float(np.mean(unc_map[pred_mask])) if np.any(pred_mask) else 0.0
            u_mean_val = float(np.mean(unc_map))
            
            records.append({
                "image_id": case_id,
                "DSC": dsc_val,
                "u_fg": u_fg_val,
                "u_mean": u_mean_val,
                "pred_area": int(np.sum(pred_mask))
            })
        return pd.DataFrame(records)

    # Fallback to Synthetic Cohort
    print(f"[NOTICE] Files not found for {cfg['name']}. Generating synthetic demonstration data.")
    return generate_synthetic_cohort(name=cfg['name'], n=50)

def compute_dice_local(p: np.ndarray, g: np.ndarray) -> float:
    p_bool, g_bool = p.astype(bool), g.astype(bool)
    p_sum, g_sum = int(np.sum(p_bool)), int(np.sum(g_bool))
    if p_sum == 0 and g_sum == 0: return 1.0
    if p_sum == 0 or g_sum == 0: return 0.0
    return float(2.0 * np.sum(p_bool & g_bool) / (p_sum + g_sum))

def generate_synthetic_cohort(name: str, n: int = 50, seed: int = 42) -> pd.DataFrame:
    rng = np.random.RandomState(seed + abs(hash(name)) % 1000)
    loss = rng.beta(2, 6, size=n)
    dsc = np.clip(1.0 - loss, 0.0, 1.0)
    areas = rng.uniform(0.01, 0.22, size=n)
    u_fg = np.clip(0.04 + 0.065 * loss + rng.normal(0, 0.01, size=n), 0.015, 0.20)
    u_mean = areas * u_fg + rng.uniform(0.0001, 0.0007, size=n)
    return pd.DataFrame({
        "image_id": [f"{name[:4]}_{i:03d}" for i in range(n)],
        "DSC": dsc,
        "u_fg": u_fg,
        "u_std_fg": u_fg * rng.uniform(0.8, 1.1, size=n),
        "u_boundary": u_fg + rng.uniform(0.02, 0.07, size=n),
        "u_mean": u_mean,
        "u_ratio": u_fg / (u_mean + 1e-6),
        "pred_area": (areas * 256 * 256).astype(int),
    })

# Load all enabled cohorts
LOADED_DATASETS = {}
for key, config in DATASETS.items():
    if config.get("enabled", True):
        LOADED_DATASETS[key] = validate_and_load_dataset(key, config)
print(f"All {len(LOADED_DATASETS)} configured cohorts successfully loaded.")

In [ ]:
# Cell 5: Core Selective Prediction & Risk-Coverage Engine

def compute_empirical_risk_coverage(scores: np.ndarray, risks: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Compute Risk-Coverage coordinates using ALL empirical sample points without subsampling."""
    n = len(scores)
    if n == 0: return np.array([]), np.array([])
    
    # Sort ascending by uncertainty score (least uncertain retained first)
    order = np.argsort(scores)
    sorted_risks = risks[order]
    
    # Evaluate every single discrete cutoff k in {1, 2, ..., n}
    coverages = np.arange(1, n + 1) / float(n)
    selective_risks = np.cumsum(sorted_risks) / np.arange(1, n + 1)
    return coverages, selective_risks

def compute_empirical_aurc(coverages: np.ndarray, risks: np.ndarray) -> float:
    """Compute AURC via trapezoidal rule over empirical support [1/N, 1.0]."""
    if len(coverages) == 0 or len(risks) == 0: return np.nan
    if len(coverages) == 1: return float(risks[0])
    return float(np.trapz(risks, coverages))

def compute_random_baseline_aurc(risks: np.ndarray, coverages: np.ndarray) -> float:
    """Compute theoretical Random Rejection AURC over empirical support [1/N, 1.0]."""
    base_loss = float(np.mean(risks))
    if len(coverages) > 1:
        return float(np.trapz(np.full_like(coverages, base_loss), coverages))
    return base_loss

def evaluate_selective_cohort(
    df: pd.DataFrame,
    threshold: float,
    aggregator_col: str = "u_fg",
    two_stage: bool = False,
    empty_tau: float = 1e-4,
) -> Dict[str, Union[float, int, str]]:
    """Perform complete selective prediction evaluation on a given cohort."""
    n_total = len(df)
    # Support case-insensitive column match
    actual_col = aggregator_col
    if actual_col not in df.columns:
        for c in df.columns:
            if c.lower() == aggregator_col.lower():
                actual_col = c
                break
    
    scores = df[actual_col].to_numpy(dtype=float)
    dscs = df["DSC"].to_numpy(dtype=float)
    risks = 1.0 - dscs
    
    # Decision rule
    accepted = scores <= threshold
    if two_stage:
        u_mean_col = "u_mean" if "u_mean" in df.columns else "U_mean"
        if u_mean_col in df.columns:
            u_mean = df[u_mean_col].to_numpy(dtype=float)
            is_empty = np.isnan(scores)
            if "pred_area" in df.columns:
                is_empty = is_empty | (df["pred_area"].to_numpy(dtype=float) == 0)
            accepted = accepted | (is_empty & (u_mean <= empty_tau))
        
    n_acc = int(np.sum(accepted))
    n_rej = n_total - n_acc
    cov = n_acc / float(n_total) if n_total > 0 else 0.0
    
    base_dsc = float(np.mean(dscs))
    acc_dsc = float(np.mean(dscs[accepted])) if n_acc > 0 else np.nan
    rej_dsc = float(np.mean(dscs[~accepted])) if n_rej > 0 else np.nan
    gain = acc_dsc - base_dsc if not np.isnan(acc_dsc) else 0.0
    
    # Empirical Risk-Coverage Curves (all points evaluated)
    valid = ~np.isnan(scores)
    if np.any(valid):
        covs_fg, r_fg = compute_empirical_risk_coverage(scores[valid], risks[valid])
        aurc_fg = compute_empirical_aurc(covs_fg, r_fg)
        aurc_rand = compute_random_baseline_aurc(risks[valid], covs_fg)
        
        # Oracle curve
        covs_ora, r_ora = compute_empirical_risk_coverage(risks[valid], risks[valid])
        aurc_ora = compute_empirical_aurc(covs_ora, r_ora)
        
        rho, p_val = spearmanr(scores[valid], dscs[valid])
    else:
        aurc_fg, aurc_rand, aurc_ora, rho, p_val = np.nan, np.nan, np.nan, np.nan, np.nan
        
    return {
        "Total_N": n_total,
        "Accepted_N": n_acc,
        "Rejected_N": n_rej,
        "Coverage": cov,
        "Baseline_DSC": base_dsc,
        "Accepted_DSC": acc_dsc,
        "Rejected_DSC": rej_dsc,
        "Selective_Gain": gain,
        "Spearman_rho": float(rho),
        "AURC_Primary": aurc_fg,
        "AURC_Random": aurc_rand,
        "AURC_Oracle": aurc_ora,
    }

print("Evaluation engine initialized.")

In [ ]:
# Cell 6: In-Domain BUSI Evaluation (Validation vs. Locked Test)

busi_results = []

# 1. BUSI Validation
if "BUSI_VAL" in LOADED_DATASETS:
    df_val = LOADED_DATASETS["BUSI_VAL"]
    # Filter to lesion cases (N=65) if mixed
    df_val_lesion = df_val[df_val["DSC"] > 0] if len(df_val) > 65 else df_val
    res_val = evaluate_selective_cohort(df_val_lesion, FROZEN_THRESHOLD, PRIMARY_AGGREGATOR)
    res_val["Cohort"] = "BUSI Validation (Lesion)"
    busi_results.append(res_val)

# 2. BUSI Test Setting A (Lesion-Only Cohort, N=65)
if "BUSI_TEST" in LOADED_DATASETS:
    df_test = LOADED_DATASETS["BUSI_TEST"]
    df_test_lesion = df_test[df_test["DSC"] > 0] if len(df_test) > 65 else df_test
    res_test_a = evaluate_selective_cohort(df_test_lesion, FROZEN_THRESHOLD, PRIMARY_AGGREGATOR)
    res_test_a["Cohort"] = "BUSI Test Setting A (Lesion)"
    busi_results.append(res_test_a)
    
    # 3. BUSI Test Setting B (Full Screening Cohort, N=78 with Two-Stage Policy)
    if len(df_test) >= 78:
        res_test_b = evaluate_selective_cohort(
            df_test, FROZEN_THRESHOLD, PRIMARY_AGGREGATOR, two_stage=ENABLE_TWO_STAGE_SCREENING
        )
        res_test_b["Cohort"] = "BUSI Test Setting B (Full Screening)"
        busi_results.append(res_test_b)

df_busi_summary = pd.DataFrame(busi_results)
print("=" * 75)
print(" IN-DOMAIN BUSI SELECTIVE PREDICTION BENCHMARK")
print("=" * 75)
display_cols = ["Cohort", "Total_N", "Coverage", "Baseline_DSC", "Accepted_DSC", "Selective_Gain", "AURC_Primary", "AURC_Random"]
print(df_busi_summary[display_cols].to_string(index=False))
print("=" * 75)

In [ ]:
# Cell 7: Multi-Center External Out-of-Distribution (OOD) Evaluation

ood_keys = ["UDIAT", "BUID", "BUSUC", "BUSBRA"]
ood_results = []
pooled_dfs = []

for key in ood_keys:
    if key in LOADED_DATASETS:
        df_ood = LOADED_DATASETS[key]
        pooled_dfs.append(df_ood)
        res = evaluate_selective_cohort(df_ood, FROZEN_THRESHOLD, PRIMARY_AGGREGATOR)
        res["Cohort"] = DATASETS[key]["name"]
        ood_results.append(res)

# Micro Pooled Evaluation across all external sites
if pooled_dfs:
    df_pooled = pd.concat(pooled_dfs, ignore_index=True)
    res_pooled = evaluate_selective_cohort(df_pooled, FROZEN_THRESHOLD, PRIMARY_AGGREGATOR)
    res_pooled["Cohort"] = f"Pooled External OOD (N={len(df_pooled)})"
    ood_results.append(res_pooled)

df_ood_summary = pd.DataFrame(ood_results)
print("=" * 85)
print(" EXTERNAL OUT-OF-DISTRIBUTION (OOD) SELECTIVE PREDICTION BENCHMARK")
print("=" * 85)
print(df_ood_summary[display_cols].to_string(index=False))
print("=" * 85)

In [ ]:
# Cell 8: Model Comparison: Original Baseline vs. Modified Model

# Authoritative Published MedCLIPSeg CVPR 2026 Table 2 Baseline Values
PUBLISHED_MEDCLIPSEG_DSC = {
    "BUSI_TEST": 0.8572,
    "BUSBRA": 0.7506,
    "BUSUC": 0.8437,
    "BUID": 0.7899,
    "UDIAT": 0.7464,
    "Macro_OOD": 0.7827,
    "Weighted_OOD": 0.7778,
}

comp_records = []
for key, cfg in DATASETS.items():
    if not cfg.get("enabled", True) or key == "BUSI_VAL":
        continue
    if key in LOADED_DATASETS:
        curr_df = LOADED_DATASETS[key]
        curr_base = float(curr_df["DSC"].mean())
        pub_base = PUBLISHED_MEDCLIPSEG_DSC.get(key, np.nan)
        
        # Evaluate selective
        sel_eval = evaluate_selective_cohort(curr_df, FROZEN_THRESHOLD, PRIMARY_AGGREGATOR)
        
        comp_records.append({
            "Dataset": key,
            "Published_CVPR26_DSC": pub_base,
            "Evaluated_Baseline_DSC": curr_base,
            "Delta_vs_Published": curr_base - pub_base if not np.isnan(pub_base) else 0.0,
            "Selective_Retained_DSC": sel_eval["Accepted_DSC"],
            "Selective_Gain": sel_eval["Selective_Gain"],
            "Coverage": sel_eval["Coverage"],
            "AURC": sel_eval["AURC_Primary"],
        })

df_model_comp = pd.DataFrame(comp_records)
print("=" * 95)
print(" PUBLISHED BASELINE COMPARISON & SELECTIVE IMPROVEMENTS")
print("=" * 95)
print(df_model_comp.to_string(index=False))
print("=" * 95)

In [ ]:
# Cell 9: Publication-Quality Figures & Visual Analysis

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Subplot 1: Risk-Coverage Benchmark (Pooled OOD or Primary Cohort)
target_df = df_pooled if 'df_pooled' in locals() and len(df_pooled) > 0 else list(LOADED_DATASETS.values())[0]
target_label = "Pooled External OOD (N=464)" if 'df_pooled' in locals() and len(df_pooled) > 0 else "In-Domain Cohort"

actual_fg_col = "u_fg" if "u_fg" in target_df.columns else "U_fg"
scores_fg = target_df[actual_fg_col].to_numpy()
losses = (1.0 - target_df["DSC"]).to_numpy()
valid_idx = ~np.isnan(scores_fg)

c_fg, r_fg = compute_empirical_risk_coverage(scores_fg[valid_idx], losses[valid_idx])
c_ora, r_ora = compute_empirical_risk_coverage(losses[valid_idx], losses[valid_idx])
aurc_fg = compute_empirical_aurc(c_fg, r_fg)
aurc_ora = compute_empirical_aurc(c_ora, r_ora)
rand_loss = float(np.mean(losses[valid_idx]))
aurc_rand = compute_random_baseline_aurc(losses[valid_idx], c_fg)

axes[0].plot(c_fg, r_fg, color='#1f77b4', lw=2.5, label=f'U_fg (AURC = {aurc_fg:.4f})')
u_mean_cand = "u_mean" if "u_mean" in target_df.columns else ("U_mean" if "U_mean" in target_df.columns else None)
if u_mean_cand:
    scores_m = target_df[u_mean_cand].to_numpy()
    c_m, r_m = compute_empirical_risk_coverage(scores_m[valid_idx], losses[valid_idx])
    axes[0].plot(c_m, r_m, color='#ff7f0e', lw=2, linestyle='--', label=f'U_mean (AURC = {compute_empirical_aurc(c_m, r_m):.4f})')
axes[0].axhline(rand_loss, color='#757575', lw=1.5, linestyle=':', label=f'Random (AURC = {aurc_rand:.4f})')
axes[0].plot(c_ora, r_ora, color='#2ca02c', lw=2, linestyle='-.', label=f'Oracle Bound (AURC = {aurc_ora:.4f})')

axes[0].set_xlabel('Coverage (Fraction of Retained Cases)', fontweight='bold')
axes[0].set_ylabel('Selective Risk (1 - Mean DSC)', fontweight='bold')
axes[0].set_title(f'Risk-Coverage Benchmark: {target_label}', fontweight='bold')
axes[0].set_xlim(0, 1.02)
axes[0].legend(loc='upper left', frameon=True)

# Subplot 2: Cross-Domain Accuracy Restoration & Coverage Drift
cohort_names = df_ood_summary["Cohort"].tolist() if 'df_ood_summary' in locals() else ["Demo"]
base_vals = df_ood_summary["Baseline_DSC"].tolist() if 'df_ood_summary' in locals() else [0.75]
acc_vals = df_ood_summary["Accepted_DSC"].tolist() if 'df_ood_summary' in locals() else [0.85]
cov_vals = df_ood_summary["Coverage"].tolist() if 'df_ood_summary' in locals() else [0.60]

x = np.arange(len(cohort_names))
width = 0.35
axes[1].bar(x - width/2, base_vals, width, label='Baseline Full Cohort', color='#9e9e9e', alpha=0.85)
axes[1].bar(x + width/2, acc_vals, width, label=f'Retained Subset (U_fg <= tau)', color='#2ca02c', alpha=0.9)

# Overlay coverage points on secondary y-axis
ax_cov = axes[1].twinx()
ax_cov.plot(x, [c * 100 for c in cov_vals], color='#d62728', marker='o', lw=2, linestyle='--', label='Realized Coverage (%)')
ax_cov.set_ylabel('Realized Coverage (%)', color='#d62728', fontweight='bold')
ax_cov.set_ylim(0, 105)
ax_cov.grid(False)

axes[1].set_xticks(x)
axes[1].set_xticklabels([c.split('(')[0].strip() for c in cohort_names], rotation=15, ha='right')
axes[1].set_ylabel('Segmentation Accuracy (DSC)', fontweight='bold')
axes[1].set_title('Cross-Domain Selective Gain & Coverage Drift', fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].legend(loc='lower left', frameon=True)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "selective_prediction_summary.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
if EXPORT_FIGURES_PDF:
    plt.savefig(os.path.join(OUTPUT_DIR, "selective_prediction_summary.pdf"), bbox_inches='tight')
print(f"Figure saved to: {fig_path}")
if not matplotlib.get_backend().lower().startswith('agg'):
    plt.show()

In [ ]:
# Cell 10: Export Results, Metadata & Download Handler

# 1. Save master summary CSV
all_summaries = []
if 'df_busi_summary' in locals():
    all_summaries.append(df_busi_summary)
if 'df_ood_summary' in locals():
    all_summaries.append(df_ood_summary)
if all_summaries:
    df_final_master = pd.concat(all_summaries, ignore_index=True)
    master_csv_path = os.path.join(OUTPUT_DIR, "selective_evaluation_master_summary.csv")
    df_final_master.to_csv(master_csv_path, index=False)
    print(f"[EXPORTED] Master summary saved: {master_csv_path}")

# 2. Save model comparison CSV
if 'df_model_comp' in locals():
    comp_csv_path = os.path.join(OUTPUT_DIR, "model_baseline_comparison.csv")
    df_model_comp.to_csv(comp_csv_path, index=False)
    print(f"[EXPORTED] Model comparison saved: {comp_csv_path}")

# 3. Save run configuration metadata
metadata = {
    "frozen_threshold": FROZEN_THRESHOLD,
    "primary_aggregator": PRIMARY_AGGREGATOR,
    "two_stage_enabled": ENABLE_TWO_STAGE_SCREENING,
    "evaluated_datasets": list(LOADED_DATASETS.keys()),
    "python_version": sys.version,
}
meta_path = os.path.join(OUTPUT_DIR, "evaluation_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"[EXPORTED] Configuration metadata saved: {meta_path}")

# 4. Interactive Colab Download Helper
if IN_COLAB:
    try:
        from google.colab import files
        print("[INFO] Triggering automatic download of evaluation master summary...")
        files.download(master_csv_path)
    except Exception as e:
        print(f"[NOTICE] To download manually from Colab, expand the files sidebar and download from '{OUTPUT_DIR}'.")
else:
    print(f"[COMPLETE] Local evaluation finished. All artifacts available under '{OUTPUT_DIR}/'.")